# 03 — Test Matching

Test queries against the built FAISS index and tune similarity thresholds.

**Run this on MAESTRO SageMaker JupyterLab.**  
Prerequisite: Run `02_run_indexing.ipynb` first.

## 1. Setup

In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from matching.pipeline import match_entities
from matching.search import load_index

# Pre-load the index (downloads from S3 on first call)
index, metadata = load_index()
print(f"Index loaded: {index.ntotal} vectors")
print(f"Metadata loaded: {len(metadata)} rows")

## 2. Test with single entity names

In [ ]:
# Try a few entity names — edit these to match real companies you want to test
test_names = [
    "ABC TRADING PTE LTD",
    "SINGAPORE AIRLINES",
    "DBS BANK",
    "GRAB HOLDINGS",
    "SOME NONEXISTENT COMPANY XYZ",
]

results = match_entities(test_names)
results

## 3. Inspect results per query

In [ ]:
for name in test_names:
    subset = results[results["query_name"] == name]
    print(f"\n{'='*60}")
    print(f"Query: {name}")
    print(f"{'='*60}")
    if subset["matched_entity"].isna().all():
        print("  No matches above threshold.")
    else:
        for _, row in subset.iterrows():
            print(f"  #{int(row['rank'])}  {row['matched_entity']}  (score: {row['score']:.4f})")

## 4. Experiment with different thresholds

The default threshold is set in `config.py`. Try different values here to find the sweet spot.

In [ ]:
from matching.search import search

# Get raw candidates (before threshold filtering) for a single query
query = "DBS BANK"
candidates = search([query], top_k=20)

print(f"Top-20 candidates for '{query}':")
print(f"{'Rank':<6} {'Score':<10} {'Entity Name'}")
print("-" * 60)
for i, c in enumerate(candidates[0], start=1):
    marker = "  " if c["score"] >= 0.70 else "< "
    print(f"{i:<6} {c['score']:<10.4f} {marker}{c['entity_name']}")

print()
print("Rows marked with '<' are below the 0.70 default threshold.")
print("Adjust SIMILARITY_THRESHOLD in config.py based on what you see.")

## 5. Test with a CSV batch

In [ ]:
import pandas as pd

# Create a small test CSV — replace with a real file if you have one
test_df = pd.DataFrame({"company_name": test_names})
query_names = test_df["company_name"].dropna().str.strip().tolist()

batch_results = match_entities(query_names)
print(f"Input: {len(query_names)} queries")
print(f"Output: {len(batch_results)} result rows")
print(f"Queries with at least one match: {batch_results[batch_results['matched_entity'].notna()]['query_name'].nunique()}")
print()
batch_results

## Next step

Once you're happy with the matching quality, proceed to **`04_deploy_endpoint.ipynb`** to deploy the SageMaker endpoint.